# Chronos-2 기반 시계열 이상 탐지 (변수별, multivariate)

**아이디어**: 품질 변수와 POLYCOM 운전시간을 제외한 공정 변수 38개를 Chronos-2에 함께 입력하고, 과거 168시간으로 1시간 뒤를 예측합니다. 실제값이 예측 분포의 1~99% 구간 밖이면 이상으로 표시합니다.

기존 `chronos-bolt-small`(univariate, 변수 하나만 보고 그 변수를 예측)과 달리, **Chronos-2는 여러 변수를 함께 입력하면 서로의 패턴을 참고해서 예측**합니다 — 예: FEED량은 정상인데 그에 맞춰 같이 움직여야 할 온도가 안 움직이는 경우도 잡을 수 있습니다.

**사용 방법**
1. `CONFIG` 셀에서 `EXCEL_PATH`/`SHEET_NAME`이 맞는지 확인 (현재 2CM 운전 데이터로 설정됨)
2. 전처리는 GitHub Cement `review_v2`처럼 물리 한계 위반값만 NaN 처리하고, IQR/주변 중앙값 치환은 하지 않음
3. `QUICK_TEST_ROWS`로 소규모 확인 후 `None`으로 바꿔 전체 재실행
4. Full 파인튜닝은 시간순 70/15/15 분할 셀부터 순서대로 실행

**환경 확인 결과**: `amazon/chronos-2` 모델이 이 Mac에서 MPS(Apple GPU)로 정상 로드/추론되는 것 확인했습니다. Apache-2.0 라이선스 오픈소스 모델이라 비용 없이 사용 가능합니다.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from chronos import BaseChronosPipeline

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

# 기본 폰트(DejaVu Sans)에는 한글 글리프가 없어서 컬럼명의 한글이 깨져(□) 보이는 문제 방지.
# 맥/윈도우 둘 다에서 돌릴 수 있게, 실제 설치된 폰트 목록에서 있는 것만 골라서 씀
# (없는 폰트를 그냥 지정하면 matplotlib이 기본 폰트로 조용히 대체해버려서 한글이 깨짐).
import matplotlib.font_manager as fm

_korean_font_candidates = ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"]
_available_fonts = {f.name for f in fm.fontManager.ttflist}
_korean_font = next((f for f in _korean_font_candidates if f in _available_fonts), None)
if _korean_font:
    plt.rcParams["font.family"] = _korean_font
    print(f"한글 폰트 사용: {_korean_font}")
else:
    print("경고: 한글 폰트(AppleGothic/맑은 고딕/나눔고딕)를 못 찾았습니다 -- 그래프의 한글이 깨질 수 있습니다.")
plt.rcParams["axes.unicode_minus"] = False  # 일부 한글 폰트엔 유니코드 마이너스(−) 글리프가 없어서 꺼줌


In [ ]:
# ===================== CONFIG: GitHub Cement review_v2 방식 =====================
from pathlib import Path

from cement_chronos_review_v2 import (
    DOWNTIME_COL, PROTOCOL_VERSION, QUALITY_COLS,
    chronological_split, equipment_frame, file_sha256,
    load_preprocessed_equipment, loss_history_from_callback,
    make_loss_history_callback, rolling_forecast_multivariate as rolling_forecast_review_v2,
    runtime_versions, to_chronos_inputs, validation_windows,
)

EXCEL_PATH = "운전데이터_통합_2CM_3CM_4CM.xlsx"
SHEET_NAME = "운전데이터_통합"
HEADER_ROW = 0
EQUIPMENT_IDS = ["2CM", "3CM", "4CM"]

# 예측 대상에서 제외하는 건 운전시간/품질 2개뿐이다 (관측률과 무관하게 나머지는 전부 예측).
#   POLYCOM 운전시간: 정지 판정/past covariate로만 사용, 예측 대상 아님
#   기타 품질 BLAINE, 44마이크론R: 품질 지표, 예측 대상 아님
#   기타 분쇄조제: 제어변수라서 다시 예측 대상에 포함시킴 (제어변수+모니터링변수를 다 예측하기로 함)
#   기타 품종/비고: 범주형이라 load_preprocessed_equipment 안에서 이미 드롭되어 target_cols에
#   아예 안 들어옴 -- 그래서 아래 목록엔 명시적으로 안 넣어도 됨.
PREDICT_EXCLUDE_COLS = [DOWNTIME_COL, *QUALITY_COLS]
GLITCH_VARIANTS = []                 # 기존 중앙값 글리치 분기는 실행하지 않음

CONTEXT_LENGTH = 168                # 과거 1주일
PREDICTION_LENGTH = 1               # 요청대로 1시간 뒤 하나만 예측
STRIDE = 1
assert PREDICTION_LENGTH == 1, "현재 review_v2 파이프라인은 prediction_length=1로 고정합니다."

ANOMALY_QUANTILE_LOW = 0.01
ANOMALY_QUANTILE_HIGH = 0.99
SEVERITY_EPS = 1e-3

# 물리기준 정리(하드코딩)는 항상 적용. 그 위에 추가로 상위/하위 10% percentile 정리를
# 적용한 버전도 같이 만들어서 두 개를 비교합니다 (GitHub 원본은 IQR 자체를 안 씀 --
# blaine/residue 둘 다 A/B해서 "IQR 미적용이 더 낫다"는 결론으로 canonical 파이프라인에서
# 뺐음. 저희는 그래도 직접 비교해보고 싶어서 별도 버전으로 만드는 것).
# False: 물리기준 정리만 (run_key: {설비}__review_v2)
# True : 물리기준 정리 + 변수별/설비별 상위 10%·하위 10% 추가 NaN 처리 (run_key: {설비}__review_v2_iqr)
IQR_VARIANTS = [False, True]
IQR_LOW_PCT = 0.10
IQR_HIGH_PCT = 0.90

PREPROCESSED_DIR = Path("data/processed")
PREPROCESSED_CSV = PREPROCESSED_DIR / "process_timeseries_review_v2.csv"

MODEL_ID = "amazon/chronos-2"
BATCH_WINDOWS = 128
QUICK_TEST_ROWS = None              # None이면 전체 기간

if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"

print(f"Selected device: {DEVICE}, protocol={PROTOCOL_VERSION}, prediction_length={PREDICTION_LENGTH}")


In [ ]:
# ===================== 데이터 로드: GitHub review_v2 전처리 =====================
def load_data(equipment_id):
    """물리 기준 마스킹 -> 내수 품종 필터 -> 숫자 변환 -> 1시간 grid 재색인.
    관측률과 무관하게, PREDICT_EXCLUDE_COLS에 명시된 것만 예측 대상에서 뺀다(데이터엔 남겨둠).
    운전시간/품질은 모듈(load_preprocessed_equipment)에서 이미 target_cols 밖으로 빠져있어서
    아래 필터는 사실상 '기타 분쇄조제'에만 적용됨."""
    data, time_index, target_cols, cleaning_audit, stats = load_preprocessed_equipment(
        EXCEL_PATH, SHEET_NAME, equipment_id, header_row=HEADER_ROW,
    )
    target_cols = [c for c in target_cols if c not in PREDICT_EXCLUDE_COLS]
    stats["n_targets"] = len(target_cols)
    return data, time_index, target_cols, cleaning_audit, stats


In [ ]:
# ===================== 이상치 제거 원칙 =====================
# 기존 ±3시간 중앙값/Hampel 유사 글리치 탐지는 사용하지 않는다.
# GitHub Cement_code_fin의 canonical 방식처럼 명백한 물리/설비 한계 위반값만 NaN 처리한다.
# 따라서 값을 주변 중앙값으로 대체하지 않으며, 통계적 IQR 제거도 수행하지 않는다.
print("전처리: fixed physical limits -> NaN, IQR/median replacement 없음")


In [ ]:
# ===================== Chronos 모델 로드 =====================
pipeline = BaseChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print(f"Loaded {MODEL_ID} on {DEVICE}")


In [ ]:
# ===================== 예측 정확도 계산 함수 =====================
# gap/down/warmup 시점은 빼고 계산하고, "직전 시간값"이 idx 기준으로 진짜 1시간 전인 경우만
# 스코어링 대상에 포함합니다 (안 그러면 시간갭 바로 다음 시점을 마치 정상 연속 구간처럼 계산하는
# 오류가 생김 -- naive_pred는 이 유효성 체크에만 쓰고 지표 자체(MAE)는 안 냄).
# MAPE는 실제값이 0에 가까우면(예: 0~1 사이를 오가는 변수) 분모가 0에 가까워져서 터무니없이
# 커지는 문제가 있어서, |실제값| <= mape_min_abs인 시점은 MAPE 계산에서만 제외합니다
# (mape_n/mape_skipped로 몇 개를 뺐는지 남겨둠 -- MAPE가 이상하게 크게 나오는 변수는 이 값도 같이 확인).
# Accuracy(%) = 100 - MAPE(%) -- |측정값-참값|/참값 x 100(%)의 평균을 100에서 뺀 값으로,
# MAPE가 0에 가까울수록(=오차가 작을수록) Accuracy는 100에 가까워짐 (1세대 ANN 보고서 관례).
# 평가지표는 R²/RMSE/MAPE/Accuracy/Coverage 5개만 씀 -- MAE는 안 씀.

def compute_accuracy_summary(results_df, mape_min_abs=1e-6):
    rows = []
    for col, sub in results_df.groupby("variable"):
        sub = sub.sort_values("idx").reset_index(drop=True)
        actual = sub["actual"].to_numpy()
        pred = sub["pred_median"].to_numpy()
        scored = sub["excluded_reason"].isna().to_numpy()

        naive_pred = np.roll(actual, 1)
        naive_pred[0] = np.nan
        valid = scored & ~np.isnan(naive_pred)
        n_scored = int(valid.sum())
        if n_scored < 2:
            continue

        actual_v = actual[valid]
        err = actual_v - pred[valid]
        rmse = np.sqrt(np.mean(err ** 2))

        mape_mask = np.abs(actual_v) > mape_min_abs
        mape_n = int(mape_mask.sum())
        mape_skipped = int((~mape_mask).sum())
        mape = np.mean(np.abs(err[mape_mask] / actual_v[mape_mask])) * 100 if mape_n > 0 else np.nan
        accuracy = 100 - mape if not np.isnan(mape) else np.nan

        low = sub["pred_low"].to_numpy()[valid]
        high = sub["pred_high"].to_numpy()[valid]
        coverage = ((actual_v >= low) & (actual_v <= high)).mean() * 100

        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((actual_v - actual_v.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

        rows.append(dict(
            variable=col, label=col.replace("\n", " "),
            rmse=rmse, mape=mape, accuracy=accuracy, mape_n=mape_n, mape_skipped=mape_skipped,
            r2=r2, coverage=coverage, n_scored=n_scored,
        ))
    return pd.DataFrame(rows)


In [ ]:
# ===================== GitHub review_v2 방식 메인 파이프라인 =====================
# 1) 고정 물리 기준 위반값만 NaN 처리
# 2) 내수 품종만 유지, 실제 시간 갭은 1시간 grid의 NaN으로 유지
# 3) 품질 변수와 POLYCOM 운전시간은 예측 대상에서 제외
# 4) 운전시간은 past covariate 및 정지 판정에만 사용
# 5) 정리된 값으로 예측과 채점을 모두 수행 (물리 오류값은 성능지표에서 제외)
#
# IQR_VARIANTS로 두 버전을 다 만듭니다:
#  - False: 물리기준 정리만 (하드코딩된 규칙, GitHub 원본 방식) -> {설비}__review_v2
#  - True : 물리기준 정리 + 변수별 상위 10%·하위 10% 추가 NaN 처리 -> {설비}__review_v2_iqr
#    (물리기준으로 이미 정리된 데이터 위에 percentile 기준을 얹는 것 -- 물리기준 결과를 대체하지 않음)

def apply_percentile_trim(data, target_cols, low_pct, high_pct):
    """target_cols 각각에 대해 그 변수 자신의 관측값 기준 low_pct~high_pct 분위수 밖은 NaN 처리."""
    data = data.copy()
    n_trimmed = 0
    for col in target_cols:
        s = data[col]
        lo, hi = s.quantile(low_pct), s.quantile(high_pct)
        mask = (s < lo) | (s > hi)
        n_trimmed += int(mask.sum())
        data.loc[mask, col] = np.nan
    return data, n_trimmed


_prepared_cache = {}
_processed_frames = []
for equipment_id in EQUIPMENT_IDS:
    data_full, time_full, variable_cols, cleaning_audit, prep_stats = load_data(equipment_id)
    _prepared_cache[equipment_id] = (data_full, time_full, variable_cols, cleaning_audit, prep_stats)
    _processed_frames.append(equipment_frame(equipment_id, data_full, time_full, variable_cols))
    print(f"[{equipment_id}] {prep_stats}")

PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed_all = pd.concat(_processed_frames, ignore_index=True)
processed_all.to_csv(PREPROCESSED_CSV, index=False, encoding="utf-8-sig")
print(f"전처리 데이터 저장: {PREPROCESSED_CSV} ({len(processed_all):,}행)")

try:
    if any(r.get("protocol") != PROTOCOL_VERSION for r in equipment_results.values()):
        equipment_results = {}
except NameError:
    equipment_results = {}

for use_iqr_trim in IQR_VARIANTS:
    for equipment_id in EQUIPMENT_IDS:
        run_key = f"{equipment_id}__{'review_v2_iqr' if use_iqr_trim else 'review_v2'}"
        if run_key in equipment_results:
            print(f"{run_key}: 이미 처리됨 -> 건너뜀")
            continue

        data, time_index, variable_cols, cleaning_audit, prep_stats = _prepared_cache[equipment_id]
        if QUICK_TEST_ROWS is not None:
            data = data.tail(QUICK_TEST_ROWS).reset_index(drop=True)
            time_index = time_index.tail(QUICK_TEST_ROWS).reset_index(drop=True)

        n_iqr_trimmed = 0
        if use_iqr_trim:
            data, n_iqr_trimmed = apply_percentile_trim(data, variable_cols, IQR_LOW_PCT, IQR_HIGH_PCT)

        is_gap_row = data[variable_cols].isna().all(axis=1).to_numpy()
        is_down = data[DOWNTIME_COL].eq(0).fillna(False).to_numpy()
        is_warmup = np.zeros(len(data), dtype=bool)  # review_v2에서는 임의의 재가동 제외시간을 두지 않음
        score_eligible = ~is_gap_row & ~is_down
        variable_scale = data.loc[score_eligible, variable_cols].std().fillna(0.0).to_numpy()

        all_results = rolling_forecast_review_v2(
            pipeline, data, variable_cols, variable_scale, is_down,
            context_length=CONTEXT_LENGTH, prediction_length=PREDICTION_LENGTH,
            stride=STRIDE, batch_windows=BATCH_WINDOWS,
            quantile_low=ANOMALY_QUANTILE_LOW, quantile_high=ANOMALY_QUANTILE_HIGH,
            severity_eps=SEVERITY_EPS,
        )
        all_results["timestamp"] = [time_index[i] for i in all_results["idx"]]
        results = {col: sub.reset_index(drop=True) for col, sub in all_results.groupby("variable")}
        scored = all_results[all_results["excluded_reason"].isna()]
        summary = (
            scored.groupby("variable")
            .agg(n_points=("is_anomaly", "size"), n_anomalies=("is_anomaly", "sum"), max_severity=("severity", "max"))
            .assign(anomaly_rate=lambda frame: frame["n_anomalies"] / frame["n_points"])
            .sort_values("n_anomalies", ascending=False)
        )
        accuracy_summary = compute_accuracy_summary(all_results)
        values = data[variable_cols].to_numpy(dtype=np.float32)

        equipment_results[run_key] = dict(
            protocol=PROTOCOL_VERSION, equipment_id=equipment_id, physical_cleaned=True,
            iqr_trimmed=use_iqr_trim, clean_data_glitches=True, data=data, data_raw=data.copy(),
            time_index=time_index, variable_cols=variable_cols, is_down=is_down, is_warmup=is_warmup,
            variable_scale=variable_scale, values=values, raw_values=values.copy(),
            glitch_records=[], physical_cleaning_count=len(cleaning_audit),
            iqr_trimmed_count=n_iqr_trimmed, preprocessing_stats=prep_stats,
            all_results=all_results, results=results, summary=summary, accuracy_summary=accuracy_summary,
        )
        print(f"[{run_key}] targets={len(variable_cols)}, IQR추가마스킹={n_iqr_trimmed}, scored={len(scored):,}, "
              f"anomalies={int(scored['is_anomaly'].sum()):,} ({scored['is_anomaly'].mean()*100:.2f}%)")

print(f"완료된 review_v2 run: {list(equipment_results.keys())}")


In [ ]:
# ===================== 설비별 review_v2 결과 비교 요약 =====================
# R² / coverage / MAPE / Accuracy 기준으로 비교 (MAE는 안 씀, RMSE는 이상치 하나에도 크게 튀어서 제외)
compare_rows = []
for run_key, r in equipment_results.items():
    scored = r["all_results"][r["all_results"]["excluded_reason"].isna()]
    acc = compute_accuracy_summary(r["all_results"])
    compare_rows.append(dict(
        run=run_key,
        설비=r["equipment_id"],
        물리기준정리=r.get("physical_cleaned", False),
        IQR추가정리=r.get("iqr_trimmed", False),
        변수수=len(r["variable_cols"]),
        스코어링대상=len(scored),
        이상탐지수=int(scored["is_anomaly"].sum()),
        이상비율=scored["is_anomaly"].mean() * 100,
        물리기준마스킹건수=r.get("physical_cleaning_count", 0),
        IQR마스킹건수=r.get("iqr_trimmed_count", 0),
        평균_R2=acc["r2"].mean(),
        중앙값_R2=acc["r2"].median(),
        평균_coverage=acc["coverage"].mean(),
        평균_MAPE=acc["mape"].mean(),
        평균_Accuracy=acc["accuracy"].mean(),
    ))
equipment_comparison = pd.DataFrame(compare_rows).set_index("run")
print(equipment_comparison.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
axes[0].plot(equipment_comparison.index, equipment_comparison["이상비율"], marker="o", color="tab:blue", linewidth=1.5)
axes[0].set_title("run별 이상 비율 (%)")
axes[1].plot(equipment_comparison.index, equipment_comparison["평균_R2"], marker="o", color="tab:green", linewidth=1.5)
axes[1].set_title("run별 평균 R²")
axes[2].plot(equipment_comparison.index, equipment_comparison["평균_coverage"], marker="o", color="tab:purple", linewidth=1.5)
axes[2].axhline((ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100, color="black", linestyle="--", linewidth=1)
axes[2].set_title("run별 평균 coverage (%)")
for ax in axes:
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
plt.tight_layout()
plt.show()

equipment_comparison


In [ ]:
# ===================== 전체 run(설비 x 글리치처리 여부) CSV 한 번에 저장 =====================
# CURRENT_RUN을 매번 손으로 바꿔가며 "결과 저장"/"예측값 wide 저장" 셀을 반복하는 대신,
# equipment_results에 있는 모든 run을 한 번에 순회하면서 long/wide 두 형식 다 저장합니다.
cols_to_save = ["variable", "timestamp", "actual", "pred_median", "pred_low", "pred_high",
                "error", "severity", "is_anomaly", "excluded_reason"]

for _run_key, _r in equipment_results.items():
    _all_results = _r["all_results"]

    # long format
    _export_df = _all_results[cols_to_save].copy()
    _export_df["variable"] = _export_df["variable"].str.replace("\n", " ")
    _long_path = f"chronos_anomaly_results_{_run_key}.csv"
    _export_df.to_csv(_long_path, index=False, encoding="utf-8-sig")

    # wide format (원본 엑셀 형식, 예측 중앙값 기준)
    _wide = _all_results.copy()
    _wide["variable"] = _wide["variable"].str.replace("\n", " ")
    _wide = _wide.pivot(index="timestamp", columns="variable", values="pred_median").reset_index()
    _wide.insert(0, "ID", _r["equipment_id"])
    _wide.insert(1, "근무일자", _wide["timestamp"].dt.strftime("%Y-%m-%d"))
    _wide.insert(2, "근무시간", _wide["timestamp"].dt.hour)
    _wide = _wide.drop(columns=["timestamp"])
    _wide_path = f"chronos_predicted_원본형식_{_run_key}.csv"
    _wide.to_csv(_wide_path, index=False, encoding="utf-8-sig")

    print(f"저장 완료: {_long_path} ({len(_export_df)}행), {_wide_path} ({_wide.shape[0]}행 x {_wide.shape[1]}컬럼)")

print(f"\n총 {len(equipment_results)}개 run 저장 완료: {list(equipment_results.keys())}")


In [ ]:
# ===================== 상세 분석할 review_v2 run 선택 =====================
# CURRENT_RUN의 설비명만 바꾸면 2CM/3CM/4CM 결과를 전환할 수 있습니다.
# accuracy_summary는 캐시된 값을 안 쓰고 매번 새로 계산합니다 (Chronos 재추론 없이 순수 집계라
# 빠르고, compute_accuracy_summary 함수를 수정했을 때 캐시가 옛날 버전이라 안 바뀌는 문제를 방지).
CURRENT_RUN = "2CM__review_v2"

_r = equipment_results[CURRENT_RUN]
CURRENT_EQUIPMENT = _r["equipment_id"]  # 원래 설비명만 (파일명/ID 컬럼용)
data, data_raw, variable_cols = _r["data"], _r["data_raw"], _r["variable_cols"]
variable_scale = _r["variable_scale"]
all_results, results = _r["all_results"], _r["results"]
summary = _r["summary"]
accuracy_summary = compute_accuracy_summary(all_results)

print(f"현재 상세 보기: {CURRENT_RUN} (설비={CURRENT_EQUIPMENT}, "
      f"물리 기준 NaN 처리 {_r['physical_cleaning_count']}건)")
summary


In [ ]:
# ===================== 예측 정확도 시각화 =====================
# R² / coverage / MAPE / Accuracy 네 가지를 변수별로 꺾은선 그래프로 봅니다 (변수는 R² 높은 순 정렬).
# - R²: 변수마다 스케일이 달라서 변수 간 비교에 제일 적합 (0.5 이상=잘 맞음, 음수=평균 찍는 것보다 못함).
# - coverage: 예측 구간(1~99%)에 실제값이 들어온 비율 -- 이상탐지 판정 기준 자체가 잘 맞는지 확인
# - MAPE: 상대오차(%). 실제값이 0 근처인 변수는 분모가 작아져서 값이 커질 수 있음
#   (accuracy_summary의 mape_skipped로 몇 개 뺐는지 확인 가능).
# - Accuracy: 100 - MAPE(%), 높을수록 좋음.
# MAE/RMSE는 안 씀 -- RMSE는 극단치 하나에 완전히 흔들려서(조분량 사례: 3천만짜리 값 하나로
# RMSE가 159,422까지 튐) 변수 간 비교용 메인 지표로는 부적합하다고 판단함
# (accuracy_summary["rmse"]로 따로 확인 가능).
#
# RMSE가 전체 중앙값 대비 압도적으로 큰 변수는 (다른 그래프에서도) 따로 뺌 -- 그 변수 하나가
# 축을 다 눌러버려서 나머지 변수들끼리 비교가 안 되는 문제가 있었음. 이런 변수는 메인 그래프에서
# 빼고 별도 표로 보여줌 (accuracy_summary는 위 "상세 분석할 설비 선택" 셀에서 이미 계산되어 있음).

def plot_accuracy_summary(acc=None, outlier_rmse_multiplier=20):
    # acc=None이면 호출 시점의 최신 accuracy_summary를 씀 (함수 정의 시점 값으로
    # 고정되면 계산 함수를 고쳐도 '설비 선택' 셀을 다시 안 돌리면 옛날 값이 남는 문제가 있었음)
    acc = accuracy_summary if acc is None else acc
    target_coverage = (ANOMALY_QUANTILE_HIGH - ANOMALY_QUANTILE_LOW) * 100
    acc = acc.dropna(subset=["r2"])

    median_rmse = acc["rmse"].median()
    is_extreme = acc["rmse"] > median_rmse * outlier_rmse_multiplier
    acc_extreme = acc[is_extreme]
    acc_main = acc[~is_extreme]

    acc_sorted = acc_main.sort_values("r2", ascending=False).reset_index(drop=True)
    x = range(len(acc_sorted))

    fig, axes = plt.subplots(4, 1, figsize=(max(10, len(acc_sorted) * 0.35), 14), sharex=True)

    ax = axes[0]
    ax.plot(x, acc_sorted["r2"], marker="o", markersize=4, color="tab:green", linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title("R² (설명력) -- 0.5 이상=잘 맞음, 음수=평균 찍는 것보다 못함")
    ax.tick_params(axis="y", labelsize=8)

    ax = axes[1]
    ax.plot(x, acc_sorted["coverage"], marker="o", markersize=4, color="tab:purple", linewidth=1.2)
    ax.axhline(target_coverage, color="black", linewidth=1, linestyle="--")
    ax.set_title(f"예측구간 커버리지 % (점선={target_coverage:.0f}% 이상적)")
    ax.tick_params(axis="y", labelsize=8)

    ax = axes[2]
    ax.plot(x, acc_sorted["mape"], marker="o", markersize=4, color="tab:orange", linewidth=1.2)
    ax.set_title("MAPE (%)")
    ax.tick_params(axis="y", labelsize=8)

    ax = axes[3]
    ax.plot(x, acc_sorted["accuracy"], marker="o", markersize=4, color="tab:blue", linewidth=1.2)
    ax.axhline(100, color="black", linewidth=0.8)
    ax.set_title("Accuracy (%) = 100 - MAPE")
    ax.tick_params(axis="y", labelsize=8)

    # sharex=True를 쓰면 매트플롯립이 기본적으로 맨 아래 그래프에만 x축 라벨(변수명)을 남기고
    # 위쪽은 다 지워버림 -> 매번 맨 아래까지 스크롤해서 봐야 하는 게 불편해서, 모든 패널에
    # 변수명이 다 보이게 강제로 다시 켜줌.
    for ax in axes:
        ax.set_xticks(list(x))
        ax.set_xticklabels(acc_sorted["label"], rotation=90, fontsize=7)
        ax.tick_params(axis="x", labelbottom=True)

    plt.tight_layout()
    plt.show()

    print(f"전체 평균(아래 극단치 제외, {len(acc_main)}개 기준): "
          f"R²={acc_main['r2'].mean():.3f}(중앙값 {acc_main['r2'].median():.3f}), "
          f"coverage={acc_main['coverage'].mean():.2f}%(목표 {target_coverage:.0f}%), "
          f"MAPE={acc_main['mape'].mean():.2f}%, Accuracy={acc_main['accuracy'].mean():.2f}%")
    print(f"R² >= 0.5인 변수: {(acc_main['r2'] >= 0.5).sum()}/{len(acc_main)}")

    if len(acc_extreme):
        print(f"\n[그래프에서 제외한 극단치 변수 {len(acc_extreme)}개] "
              f"RMSE가 전체 중앙값({median_rmse:.2f})의 {outlier_rmse_multiplier}배 초과 "
              f"-- 데이터 자체에 물리적으로 불가능한 극단값이 섞여있을 가능성이 높음, 별도 확인 필요:")
        display(acc_extreme[["label", "r2", "coverage", "mape", "accuracy", "n_scored"]])

    print("\n변수별 상세 (R² 높은 순, 극단치 제외):")
    display(acc_sorted[["label", "r2", "coverage", "mape", "accuracy", "n_scored"]])


plot_accuracy_summary()


In [ ]:
# ===================== 변수별 자기상관(ACF) 분석 =====================
# 각 변수가 "자기 자신의 과거값과 얼마나 관련 있는지"(자기상관, autocorrelation)를 봅니다.
# ACF가 낮으면(0 근처) 그 변수는 과거값만으로는 예측하기 어렵다는 뜻 -- Chronos R²가 낮게 나온
# 변수와 실제로 자기상관도 낮은지 서로 대조해볼 수 있음.
#
# 시간 갭(reindex로 NaN 처리된 부분)이 있어서 pandas의 `.dropna().autocorr(lag=k)`는 쓰면 안 됨
# (dropna를 먼저 하면 배열이 압축되면서 "lag k"가 실제 시간 간격과 안 맞게 됨 -- 실제로 확인된 버그).
# 대신 압축하지 않고 "실제 시간 위치" 기준으로 두 값이 모두 있는 쌍만 골라서(pairwise deletion)
# 상관계수를 계산합니다.

def compute_acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """x: 1시간 grid 기준 값 배열(NaN 포함 가능). 반환: lag 0~max_lag의 ACF 배열."""
    acf = np.full(max_lag + 1, np.nan)
    for k in range(max_lag + 1):
        if k == 0:
            acf[k] = 1.0
            continue
        a, b = x[k:], x[:-k]
        mask = ~np.isnan(a) & ~np.isnan(b)
        if mask.sum() < 10:
            continue
        acf[k] = np.corrcoef(a[mask], b[mask])[0, 1]
    return acf


ACF_MAX_LAG = 336  # 2주 (24 * 14)

_acf_by_var = {col: compute_acf(data[col].to_numpy(dtype=np.float64), ACF_MAX_LAG) for col in variable_cols}
acf_lag1 = pd.Series({col: v[1] for col, v in _acf_by_var.items()}, name="acf_lag1")
print(f"[{CURRENT_EQUIPMENT}] 변수 {len(acf_lag1)}개 자기상관(lag 0~{ACF_MAX_LAG}h) 계산 완료")


def plot_acf_overview(acf_series=None):
    acf_series = acf_lag1 if acf_series is None else acf_series
    s = acf_series.dropna().sort_values(ascending=False)
    labels = [c.replace("\n", " ") for c in s.index]

    fig, ax = plt.subplots(figsize=(max(10, len(s) * 0.35), 4))
    ax.plot(range(len(s)), s.values, marker="o", markersize=4, color="teal", linewidth=1.2)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(0.5, color="gray", linewidth=0.8, linestyle="--")
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_title(f"[{CURRENT_EQUIPMENT}] 변수별 lag-1 자기상관(ACF) "
                 f"-- 높을수록 바로 직전 값 하나로도 다음 값 예측이 쉬움")
    plt.tight_layout()
    plt.show()


def plot_acf_detail(col, max_lag=ACF_MAX_LAG):
    label = col.replace("\n", " ")
    acf = _acf_by_var[col]
    n = data[col].notna().sum()
    ci = 1.96 / np.sqrt(n)

    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.bar(range(max_lag + 1), acf, width=0.8, color="tab:blue")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axhline(ci, color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(-ci, color="gray", linestyle="--", linewidth=0.8)
    for k in (24, 168, 336):
        if k <= max_lag:
            ax.axvline(k, color="lightgray", linewidth=1, zorder=0)
    ax.set_title(f"{label} -- ACF(lag 0~{max_lag}h), 점선=95% 신뢰구간(±1.96/sqrt(n)), "
                 f"회색 세로선=24h/168h/336h")
    ax.set_xlabel("lag (시간)")
    plt.tight_layout()
    plt.show()


plot_acf_overview()

# R² 높은/낮은 변수 몇 개를 골라 자기상관 구조를 자세히 대조 (accuracy_summary는 위에서 이미 계산됨)
_acf_top = accuracy_summary.sort_values("r2", ascending=False)["variable"].head(2).tolist()
_acf_bottom = accuracy_summary.sort_values("r2", ascending=True)["variable"].head(2).tolist()
print("\nR² 높은 변수(예측 잘 됨) vs 낮은 변수(예측 잘 안 됨)의 자기상관 구조 비교:")
for _col in _acf_top + _acf_bottom:
    plot_acf_detail(_col)


# Chronos-2 Full 파인튜닝: GitHub review_v2 재현

- 예측 길이: **1시간** (`prediction_length=1`)
- 대상: 품질 변수와 `POLYCOM 운전시간`을 제외한 공정 변수 38개
- 입력 보조 변수: `POLYCOM 운전시간`은 past covariate로만 사용
- 분할: 2CM·3CM·4CM 각각 시간순 **70% 학습 / 15% 검증 / 15% 테스트**
- 검증: 각 설비의 검증 기간 전반에서 최대 128개 윈도우를 결정론적으로 선택
- 파인튜닝: `full`, learning rate `1e-6`, 1,000 steps, batch size 64, seed 42
- 테스트 구간은 학습과 모델 선택에 사용하지 않고 마지막 비교에만 사용

Full 파인튜닝은 시간이 오래 걸리므로 아래 셀을 순서대로 직접 실행합니다. 출력 폴더가 이미 비어 있지
않으면 기존 체크포인트를 덮어쓰지 않고 오류를 냅니다.


In [ ]:
# ===================== Full 파인튜닝 CONFIG =====================
FT_TRAIN_FRAC = 0.70
FT_VAL_FRAC = 0.15
FT_TEST_FRAC = 0.15
FT_CONTEXT_LENGTH = CONTEXT_LENGTH
FT_PREDICTION_LENGTH = PREDICTION_LENGTH
FT_VALIDATION_WINDOWS_PER_EQUIPMENT = 128
FT_NUM_STEPS = 1000
FT_BATCH_SIZE = 64
FT_LEARNING_RATE = 1e-6
FT_MODE = "full"
FT_SEED = 42
FT_OUTPUT_DIR = Path("checkpoints") / PROTOCOL_VERSION / "all_process_full_pred1"
FINETUNED_MODEL_DIR = FT_OUTPUT_DIR / "final"

assert FT_PREDICTION_LENGTH == 1
print(f"mode={FT_MODE}, pred_len={FT_PREDICTION_LENGTH}, context={FT_CONTEXT_LENGTH}, "
      f"steps={FT_NUM_STEPS}, batch={FT_BATCH_SIZE}, lr={FT_LEARNING_RATE}, seed={FT_SEED}")


In [ ]:
# ===================== 시간순 70/15/15 분할 + 검증 윈도우 구성 =====================
if PREPROCESSED_CSV.exists():
    ft_full_df = pd.read_csv(PREPROCESSED_CSV, parse_dates=["timestamp"])
else:
    _frames = []
    for _eid in EQUIPMENT_IDS:
        _data, _time, _targets, _, _ = load_data(_eid)
        _frames.append(equipment_frame(_eid, _data, _time, _targets))
    ft_full_df = pd.concat(_frames, ignore_index=True)
    PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    ft_full_df.to_csv(PREPROCESSED_CSV, index=False, encoding="utf-8-sig")

ft_variable_cols = [c for c in ft_full_df.columns if c not in ["item_id", "timestamp", DOWNTIME_COL]]
assert not set(QUALITY_COLS).intersection(ft_variable_cols)
assert DOWNTIME_COL not in ft_variable_cols

ft_train_df, ft_val_df, ft_test_df = chronological_split(
    ft_full_df, train_frac=FT_TRAIN_FRAC, val_frac=FT_VAL_FRAC,
)
ft_val_windows_df, ft_validation_manifest = validation_windows(
    ft_train_df, ft_val_df, ft_variable_cols,
    prediction_length=FT_PREDICTION_LENGTH, context_length=FT_CONTEXT_LENGTH,
    windows_per_item=FT_VALIDATION_WINDOWS_PER_EQUIPMENT,
)
train_inputs = to_chronos_inputs(ft_train_df, ft_variable_cols, FT_PREDICTION_LENGTH)
val_inputs = to_chronos_inputs(ft_val_windows_df, ft_variable_cols, FT_PREDICTION_LENGTH)
for _window in val_inputs:
    _labels = _window["context"][:_window["n_targets"], -FT_PREDICTION_LENGTH:]
    if not torch.isfinite(_labels).any().item():
        raise ValueError("검증 윈도우의 예측 구간에 유효한 target이 없습니다.")

print(pd.DataFrame({
    "train": ft_train_df.groupby("item_id").size(),
    "validation": ft_val_df.groupby("item_id").size(),
    "test": ft_test_df.groupby("item_id").size(),
}))
print(ft_validation_manifest.groupby("item_id").agg(
    windows=("window_id", "size"), observed_labels=("observed_labels", "sum"),
    first_forecast=("forecast_start", "min"), last_forecast=("forecast_end", "max"),
))
print(f"targets={len(ft_variable_cols)}, train_inputs={len(train_inputs)}, val_inputs={len(val_inputs)}")


In [ ]:
# ===================== Full 파인튜닝 실행 + 재현 정보 저장 =====================
import json
import time
from transformers import set_seed

if FT_OUTPUT_DIR.exists() and any(FT_OUTPUT_DIR.iterdir()):
    raise FileExistsError(f"기존 실행을 덮어쓰지 않습니다: {FT_OUTPUT_DIR}")
FT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ft_validation_manifest.to_csv(FT_OUTPUT_DIR / "validation_windows.csv", index=False)

ft_metadata = {
    "protocol": PROTOCOL_VERSION,
    "status": "started",
    "model_id": MODEL_ID,
    "prediction_length": FT_PREDICTION_LENGTH,
    "context_length": FT_CONTEXT_LENGTH,
    "target_columns": ft_variable_cols,
    "past_covariates": [DOWNTIME_COL],
    "split": {"train": FT_TRAIN_FRAC, "validation": FT_VAL_FRAC, "test": FT_TEST_FRAC},
    "finetune_mode": FT_MODE,
    "learning_rate": FT_LEARNING_RATE,
    "num_steps": FT_NUM_STEPS,
    "batch_size": FT_BATCH_SIZE,
    "seed": FT_SEED,
    "source_xlsx_sha256": file_sha256(EXCEL_PATH),
    "processed_csv_sha256": file_sha256(PREPROCESSED_CSV),
    "versions": runtime_versions(),
    "validation_windows": len(ft_validation_manifest),
    "validation_observed_labels": int(ft_validation_manifest["observed_labels"].sum()),
}
(FT_OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(ft_metadata, ensure_ascii=False, indent=2), encoding="utf-8",
)

set_seed(FT_SEED)
loss_callback = make_loss_history_callback()
_ft_start = time.time()
pipeline_ft = pipeline.fit(
    inputs=train_inputs,
    prediction_length=FT_PREDICTION_LENGTH,
    validation_inputs=val_inputs,
    finetune_mode=FT_MODE,
    learning_rate=FT_LEARNING_RATE,
    num_steps=FT_NUM_STEPS,
    batch_size=FT_BATCH_SIZE,
    context_length=FT_CONTEXT_LENGTH,
    output_dir=FT_OUTPUT_DIR,
    finetuned_ckpt_name="finetuned-ckpt",
    callbacks=[loss_callback],
    seed=FT_SEED,
    data_seed=FT_SEED,
)
_ft_elapsed = time.time() - _ft_start
pipeline_ft.save_pretrained(FINETUNED_MODEL_DIR)
loss_history = loss_history_from_callback(loss_callback)
loss_history.to_csv(FT_OUTPUT_DIR / "loss_history.csv", index=False)
if loss_history["val_loss"].dropna().empty:
    raise RuntimeError("유효한 validation loss가 기록되지 않아 실행을 완료로 표시하지 않습니다.")
ft_metadata["status"] = "complete"
ft_metadata["elapsed_minutes"] = _ft_elapsed / 60
(FT_OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(ft_metadata, ensure_ascii=False, indent=2), encoding="utf-8",
)
print(f"Full 파인튜닝 완료: {FINETUNED_MODEL_DIR} ({_ft_elapsed/60:.1f}분)")


In [ ]:
# ===================== 손대지 않은 테스트 15%에서 zero-shot과 Full 모델 재예측 =====================
def evaluate_on_test(model):
    output = []
    for equipment_id in EQUIPMENT_IDS:
        full_group = ft_full_df.loc[ft_full_df["item_id"].eq(equipment_id)].sort_values("timestamp").reset_index(drop=True)
        train_group = ft_train_df.loc[ft_train_df["item_id"].eq(equipment_id)]
        test_group = ft_test_df.loc[ft_test_df["item_id"].eq(equipment_id)]
        test_start = test_group["timestamp"].min()
        test_start_idx = int(np.flatnonzero(full_group["timestamp"].ge(test_start).to_numpy())[0])
        slice_start = max(0, test_start_idx - FT_CONTEXT_LENGTH)
        eval_group = full_group.iloc[slice_start:].reset_index(drop=True)
        eval_data = eval_group[[*ft_variable_cols, DOWNTIME_COL]].copy()
        is_down = eval_data[DOWNTIME_COL].eq(0).fillna(False).to_numpy()

        train_running = ~train_group[DOWNTIME_COL].eq(0).fillna(False)
        variable_scale = train_group.loc[train_running, ft_variable_cols].std().fillna(0.0).to_numpy()
        result = rolling_forecast_review_v2(
            model, eval_data, ft_variable_cols, variable_scale, is_down,
            context_length=FT_CONTEXT_LENGTH, prediction_length=FT_PREDICTION_LENGTH,
            stride=STRIDE, batch_windows=BATCH_WINDOWS,
            quantile_low=ANOMALY_QUANTILE_LOW, quantile_high=ANOMALY_QUANTILE_HIGH,
            severity_eps=SEVERITY_EPS,
        )
        result["timestamp"] = [eval_group["timestamp"].iloc[i] for i in result["idx"]]
        result = result.loc[result["timestamp"].ge(test_start)].copy()
        result.insert(0, "equipment_id", equipment_id)
        output.append(result)
    return pd.concat(output, ignore_index=True)

all_results_zeroshot_test = evaluate_on_test(pipeline)
all_results_ft = evaluate_on_test(pipeline_ft)
print(f"zero-shot test rows={len(all_results_zeroshot_test):,}, full test rows={len(all_results_ft):,}")


In [ ]:
# ===================== 동일한 테스트 15%에서 zero-shot vs Full 비교 =====================
# naive 비교(mae_vs_naive)는 안 씀 -- 어차피 Full 파인튜닝을 무조건 진행하기로 했으므로
# "naive보다 나은지" 판정은 필요 없고, R²/RMSE/MAPE/Accuracy/Coverage로 zero-shot과
# Full을 직접 비교합니다.
def accuracy_by_equipment(result_df, model_name):
    parts = []
    for equipment_id, group in result_df.groupby("equipment_id", sort=False):
        acc = compute_accuracy_summary(group)
        acc.insert(0, "equipment_id", equipment_id)
        acc.insert(1, "model", model_name)
        parts.append(acc)
    return pd.concat(parts, ignore_index=True)

accuracy_summary_zeroshot_test = accuracy_by_equipment(all_results_zeroshot_test, "zero-shot")
accuracy_summary_ft = accuracy_by_equipment(all_results_ft, "full-finetuned")
metric_cols = ["rmse", "mape", "accuracy", "r2", "coverage", "n_scored"]
compare = accuracy_summary_zeroshot_test[["equipment_id", "variable", "label", *metric_cols]].merge(
    accuracy_summary_ft[["equipment_id", "variable", *metric_cols]],
    on=["equipment_id", "variable"], suffixes=("_zeroshot", "_finetuned"),
)

print("=== 테스트 구간 전체 변수 평균: zero-shot -> Full 파인튜닝 ===")
print(f"R²         : {compare['r2_zeroshot'].mean():.3f} -> {compare['r2_finetuned'].mean():.3f}  (높을수록 좋음)")
print(f"RMSE       : {compare['rmse_zeroshot'].mean():.3f} -> {compare['rmse_finetuned'].mean():.3f}  (낮을수록 좋음)")
print(f"MAPE(%)    : {compare['mape_zeroshot'].mean():.2f} -> {compare['mape_finetuned'].mean():.2f}  (낮을수록 좋음)")
print(f"Accuracy(%): {compare['accuracy_zeroshot'].mean():.2f} -> {compare['accuracy_finetuned'].mean():.2f}  (높을수록 좋음)")
print(f"coverage(%): {compare['coverage_zeroshot'].mean():.2f} -> {compare['coverage_finetuned'].mean():.2f}")
print(f"\nR² 개선된 설비-변수: {int((compare['r2_finetuned'] > compare['r2_zeroshot']).sum())}/{len(compare)}")

all_results_zeroshot_test.to_csv(FT_OUTPUT_DIR / "test_predictions_zeroshot.csv", index=False, encoding="utf-8-sig")
all_results_ft.to_csv(FT_OUTPUT_DIR / "test_predictions_full.csv", index=False, encoding="utf-8-sig")
compare.to_csv(FT_OUTPUT_DIR / "test_metrics_comparison.csv", index=False, encoding="utf-8-sig")

comp_plot = compare.assign(key=lambda frame: frame["equipment_id"] + " | " + frame["label"])
comp_plot = comp_plot.sort_values("r2_finetuned")
fig, ax = plt.subplots(figsize=(11, max(7, len(comp_plot) * 0.16)))
y = np.arange(len(comp_plot))
ax.barh(y - 0.2, comp_plot["r2_zeroshot"], height=0.4, color="tab:gray", label="zero-shot")
ax.barh(y + 0.2, comp_plot["r2_finetuned"], height=0.4, color="tab:blue", label="Full 파인튜닝")
ax.axvline(0, color="black", linewidth=1, linestyle="--")
ax.set_yticks(y)
ax.set_yticklabels(comp_plot["key"], fontsize=6)
ax.invert_yaxis()
ax.set_title("테스트 15% R²: zero-shot vs Full 파인튜닝 (높을수록 좋음)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

compare.sort_values(["equipment_id", "r2_finetuned"], ascending=[True, False])
